# FPN-Mamba Experiments — A100 Colab Runner

**Before starting:** Runtime → Change runtime type → A100 GPU.

**How persistence works:**
- **Code + data** come from GitHub (cloned each session — data is committed in the repo)
- **Checkpoints and results** save to Google Drive — survive Colab resets

**Run order:** cells 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9

## Cell 1 — Verify A100 GPU

In [ ]:
import torch

assert torch.cuda.is_available(), 'No GPU. Runtime → Change runtime type → GPU → A100'
gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
bf16_ok  = torch.cuda.is_bf16_supported()

print(f'GPU  : {gpu_name}')
print(f'VRAM : {vram_gb:.1f} GB')
print(f'BF16 : {bf16_ok}  (True = A100 confirmed)')

if not bf16_ok:
    print('WARNING: Not an A100. Batch sizes below may OOM — reduce batch_size if needed.')

## Cell 2 — Mount Google Drive

Only needed for saving checkpoints and results. Code and data come from GitHub, not Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_RESULTS = '/content/drive/MyDrive/fpn_mamba/experiments'
os.makedirs(DRIVE_RESULTS, exist_ok=True)

print('Drive mounted. Results will save to:', DRIVE_RESULTS)

## Cell 3 — Clone Repo (code + data) from GitHub

The dataset is committed in the repo — no separate download needed.

In [ ]:
import os, sys

REPO_URL = 'https://github.com/Tech-sam-90/fpn-inceptentionnet'
REPO_DIR = '/content/fpn-inceptentionnet'
BRANCH   = 'version_2'

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

DATA_ROOT = f'{REPO_DIR}/data'

# Verify data is present
classes = sorted(d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d)))
n_images = sum(len(os.listdir(os.path.join(DATA_ROOT, c))) for c in classes)

print(f'Repo    : {REPO_DIR}')
print(f'Branch  : {BRANCH}')
print(f'Data    : {DATA_ROOT}')
print(f'Classes : {classes}')
print(f'Images  : {n_images}')
!git log --oneline -3

## Cell 4 — Install Dependencies

In [ ]:
!pip install -q timm einops scikit-learn scipy thop pyyaml tqdm matplotlib seaborn pandas
print('Done.')

## Config Helper

In [ ]:
import yaml
from pathlib import Path

def make_config(yaml_path: str, overrides: dict) -> str:
    def _deep_update(base, patch):
        for k, v in patch.items():
            if isinstance(v, dict) and k in base:
                _deep_update(base[k], v)
            else:
                base[k] = v
    with open(yaml_path) as f:
        cfg = yaml.safe_load(f)
    _deep_update(cfg, overrides)
    out = f'/tmp/{Path(yaml_path).stem}_patched.yaml'
    with open(out, 'w') as f:
        yaml.dump(cfg, f)
    return out

# Applied to every experiment
A100_BASE = {
    'data':     {'data_root': DATA_ROOT},
    'training': {'num_workers': 4},
}

print('Config helper ready. DATA_ROOT =', DATA_ROOT)

## Cell 5 — Train InceptentionNet Baseline

Paper-exact: LR=0.005, batch=8, 40 epochs, patience=10, σ=2.0, 4× augmentation.

In [ ]:
import os

BASELINE_RUN_DIR = f'{DRIVE_RESULTS}/inceptentionnet'
BASELINE_RESULTS = f'{BASELINE_RUN_DIR}/cv_results.json'

if os.path.exists(BASELINE_RESULTS):
    print('Already done. Delete', BASELINE_RESULTS, 'to retrain.')
else:
    cfg = make_config(
        f'{REPO_DIR}/configs/inceptentionnet.yaml',
        overrides={**A100_BASE, 'output': {'run_dir': BASELINE_RUN_DIR}}
    )
    !python scripts/train.py --config {cfg}

print('Baseline results:', BASELINE_RESULTS)

## Cell 6 — Train FPN-Mamba (Full Model)

A100: batch=32, bfloat16 auto-enabled, num_workers=4.

In [ ]:
import os

FPN_RUN_DIR = f'{DRIVE_RESULTS}/fpn_mamba'
FPN_RESULTS = f'{FPN_RUN_DIR}/cv_results.json'

if os.path.exists(FPN_RESULTS):
    print('Already done. Delete', FPN_RESULTS, 'to retrain.')
else:
    cfg = make_config(
        f'{REPO_DIR}/configs/fpn_mamba.yaml',
        overrides={
            **A100_BASE,
            'training': {'batch_size': 32, 'num_workers': 4},
            'output':   {'run_dir': FPN_RUN_DIR},
        }
    )
    !python scripts/train.py --config {cfg} --baseline_results {BASELINE_RESULTS}

print('FPN-Mamba results:', FPN_RESULTS)

## Cell 7 — Ablation Study (5 variants, ~2 hrs total)

In [ ]:
import os

ABL_RUN_DIR = f'{DRIVE_RESULTS}/ablation'
ABL_RESULTS = f'{ABL_RUN_DIR}/ablation_summary.json'

cfg = make_config(
    f'{REPO_DIR}/configs/ablation.yaml',
    overrides={
        **A100_BASE,
        'training': {'batch_size': 32, 'num_workers': 4},
        'output':   {'run_dir': ABL_RUN_DIR},
    }
)
!python scripts/run_ablation.py --config {cfg}

print('Ablation summary:', ABL_RESULTS)

## Cell 8 — Statistical Comparison (instant, reads saved results)

In [ ]:
import json
from src.evaluation.stats import compare_models, print_comparison_table
from src.evaluation.metrics import summarize_folds

with open(BASELINE_RESULTS) as f: baseline = json.load(f)
with open(FPN_RESULTS)      as f: fpn      = json.load(f)

print('=== InceptentionNet ===')
for k, v in summarize_folds(baseline['fold_results']).items():
    print(f'  {k:<18}: {v["mean"]:.4f} +/- {v["std"]:.4f}')

print('\n=== FPN-Mamba ===')
for k, v in summarize_folds(fpn['fold_results']).items():
    print(f'  {k:<18}: {v["mean"]:.4f} +/- {v["std"]:.4f}')

print('\n=== Statistical Comparison ===')
table = compare_models(
    baseline['fold_results'], fpn['fold_results'],
    name_a='InceptentionNet', name_b='FPN-Mamba'
)
print_comparison_table(table, name_a='InceptentionNet', name_b='FPN-Mamba')

## Cell 9 — Ablation Table (instant)

In [ ]:
import json, pandas as pd

with open(ABL_RESULTS) as f:
    abl = json.load(f)

DISPLAY = {
    'efficientnet_only': 'EfficientNet-B2 only',
    'fpn_standard':      '+ FPN (standard 3x3)',
    'fpn_locality':      '+ LocalityMixing',
    'fpn_cross_mamba':   '+ Cross-scale Mamba',
    'fpn_mamba_full':    '+ GeM + SE  (full model)',
}
METRICS = ['accuracy', 'precision', 'recall', 'sensitivity', 'specificity', 'f1', 'auc']

rows = []
for variant, name in DISPLAY.items():
    if variant not in abl: continue
    row = {'Variant': name}
    for m in METRICS:
        mu  = abl[variant].get(m, {}).get('mean', float('nan'))
        std = abl[variant].get(m, {}).get('std',  float('nan'))
        row[m] = f'{mu:.4f} +/- {std:.4f}'
    rows.append(row)

df = pd.DataFrame(rows).set_index('Variant')
print(df.to_string())